In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [4]:
import sys
print(sys.executable)

/usr/local/python/3.12.1/bin/python3


In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
len(documents)

72

# Q2

In [5]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
index.fit(documents)

In [6]:
query = "How does the agentic loop keep calling the model until it stops?"
results = index.search(query)
results[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

# Q3

In [16]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

rag_app = RAGBase(index=index, llm_client=client, model='llama-3.3-70b-versatile')
answer, usage = rag_app.rag("How does the agentic loop keep calling the model until it stops?")
print(usage)

CompletionUsage(completion_tokens=17, prompt_tokens=7206, total_tokens=7223, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.051846542, prompt_time=0.551455839, completion_time=0.04902337, total_time=0.600479209)


In [17]:
lengths = [len(doc['content']) for doc in documents]
print(min(lengths), max(lengths), sum(lengths) / len(lengths))

1408 11676 4618.527777777777


# Q4

In [18]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(len(chunks))

295


# Q5

In [19]:
chunk_index = Index(text_fields=["content"], keyword_fields=["filename"])
chunk_index.fit(chunks)

rag_app_chunked = RAGBase(index=chunk_index, llm_client=client, model='llama-3.3-70b-versatile')
answer2, usage2 = rag_app_chunked.rag("How does the agentic loop keep calling the model until it stops?")
print(usage2.prompt_tokens)

2333


 7206 ÷ 2333 ≈ 3.09, so "3× fewer"
 

# Q6

In [22]:
# uv add toyaikit  (run in terminal first)

def search(query: str):
    """Search the course lesson chunks for relevant content matching the query."""
    return chunk_index.search(query, num_results=5)

In [23]:
from toyaikit.chat import ChatAssistant
from toyaikit.tools import Tools
from toyaikit.llm import OpenAIClient

tools = Tools()
tools.add_tool(search)

agent = ChatAssistant(
    tools=tools,
    developer_prompt="You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.",
    chat_interface=None,  # or however your lesson version invokes it non-interactively
    llm_client=OpenAIClient(client=client, model='llama-3.3-70b-versatile')
)

agent.run("How does the agentic loop work, and how is it different from plain RAG?")

TypeError: ChatAssistant.run() takes 1 positional argument but 2 were given

In [24]:
search_calls = 0

def search(query: str):
    """Search the course lesson chunks for relevant content matching the query."""
    global search_calls
    search_calls += 1
    return chunk_index.search(query, num_results=5)

tools_schema = [{
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the course lesson chunks for relevant content matching the query.",
        "parameters": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"]
        }
    }
}]

messages = [
    {"role": "system", "content": "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."},
    {"role": "user", "content": "How does the agentic loop work, and how is it different from plain RAG?"}
]

import json

while True:
    response = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=messages,
        tools=tools_schema
    )
    msg = response.choices[0].message
    messages.append(msg)

    if not msg.tool_calls:
        print(msg.content)
        break

    for tc in msg.tool_calls:
        args = json.loads(tc.function.arguments)
        result = search(**args)
        messages.append({
            "role": "tool",
            "tool_call_id": tc.id,
            "content": json.dumps(str(result))
        })

print("search calls:", search_calls)

The agentic loop is a pattern of interaction between a human and an artificial intelligence (AI) system, where the AI system receives input from the human, processes it, and generates a response. The agentic loop is different from plain RAG (Retrieve, Augment, Generate) in that it allows the AI system to make decisions about which tools to call and when to call them, rather than just generating a response based on the input.

In the agentic loop, the AI system receives a prompt from the human and uses it to decide which tools to call. The AI system then calls the tools, receives the results, and uses those results to generate a response to the human. This process can be repeated multiple times, with the AI system making decisions about which tools to call and when to call them at each step.

The agentic loop is useful for tasks that require the AI system to make decisions about which tools to use and when to use them, such as searching for information, generating text, or answering que